# 01 - InstructBLIP Zero-Shot Inference\n\nRuns `Salesforce/instructblip-vicuna-7b` on 650 test images with checkpoint/resume.

In [ ]:
!pip -q install transformers accelerate pillow pandas scikit-learn tqdm sentencepiece

In [ ]:
import os, re, ast, json\nfrom pathlib import Path\nimport pandas as pd\nimport torch\nfrom tqdm import tqdm\nfrom PIL import Image\nfrom transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration\n\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))

## Data setup\nUpload `colab_experiments/` folder to Drive and set `BASE_DIR`.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\n# CHANGE THIS if needed\nBASE_DIR = Path('/content/drive/MyDrive/categorization/colab_experiments')\nDATA_DIR = BASE_DIR / 'data'\nRESULTS_DIR = BASE_DIR / 'results'\nRESULTS_DIR.mkdir(parents=True, exist_ok=True)\n\nimport sys\nsys.path.append(str(BASE_DIR / 'utils'))\nfrom data_loader import load_metadata\nfrom prompt_templates import format_instructblip_prompt, SYMPTOM_NAMES\nfrom metrics import parse_predictions, calculate_metrics, print_results_table, save_metrics_csv

In [ ]:
metadata_df = load_metadata(DATA_DIR / 'metadata.csv')\nprint('Rows:', len(metadata_df))\nmetadata_df.head(2)

In [ ]:
model_id = 'Salesforce/instructblip-vicuna-7b'\nprocessor = InstructBlipProcessor.from_pretrained(model_id)\nmodel = InstructBlipForConditionalGeneration.from_pretrained(\n    model_id,\n    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n    device_map='auto'\n)\nmodel.eval()

In [ ]:
def _extract_pred(text: str):\n    m = re.search(r'PREDICTION:\\s*\\[([01,\\s]+)\\]', text, flags=re.IGNORECASE)\n    if not m:\n        return None\n    arr = [x.strip() for x in m.group(1).split(',')]\n    if len(arr) == 7 and all(x in ('0','1') for x in arr):\n        return '[' + ','.join(arr) + ']'\n    return None\n\nfinal_csv = RESULTS_DIR / 'instructblip_final.csv'\ncheckpoint_interval = 50\n\ndone = {}\nif final_csv.exists():\n    prev = pd.read_csv(final_csv)\n    done = {r['image_id']: r for _, r in prev.iterrows()}\n    print('Resuming from', len(done), 'rows')\n\nrows = []\nfor _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):\n    iid = row['image_id']\n    if iid in done:\n        rows.append(done[iid])\n        continue\n    try:\n        image = Image.open(DATA_DIR / row['image_path']).convert('RGB')\n        prompt = format_instructblip_prompt()\n        inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)\n        with torch.no_grad():\n            out = model.generate(**inputs, max_new_tokens=128)\n        text = processor.batch_decode(out, skip_special_tokens=True)[0]\n        pred = _extract_pred(text)\n        rec = {\n            'image_id': iid,\n            'category': row['category'],\n            'prediction_raw': text,\n            'prediction_vec': pred if pred else '[0,0,0,0,0,0,0]',\n            'parse_ok': bool(pred),\n            'human_label': row['human_label']\n        }\n    except Exception as e:\n        rec = {\n            'image_id': iid,\n            'category': row['category'],\n            'prediction_raw': f'ERROR: {e}',\n            'prediction_vec': '[0,0,0,0,0,0,0]',\n            'parse_ok': False,\n            'human_label': row['human_label']\n        }\n    rows.append(rec)\n    if len(rows) % checkpoint_interval == 0:\n        pd.DataFrame(rows).to_csv(final_csv, index=False)\n        print('checkpoint:', len(rows))\n\nres_df = pd.DataFrame(rows)\nres_df.to_csv(final_csv, index=False)\nprint('Saved:', final_csv)

In [ ]:
y_true = parse_predictions(res_df['human_label'].tolist())\ny_pred = parse_predictions(res_df['prediction_vec'].tolist())\nm = calculate_metrics(y_true, y_pred, SYMPTOM_NAMES)\nprint_results_table(m)\nsave_metrics_csv(m, RESULTS_DIR / 'instructblip_metrics.csv')\nwith open(RESULTS_DIR / 'instructblip_metrics.json', 'w') as f:\n    json.dump(m, f, indent=2)\nprint('metrics saved')